# Human Audit — classifier vs. consensus label disagreements (0.933 run)

Loads the **saved primary model** (`best_model_ours-mHuBERT-147`, the 0.933-UAR run), runs it on the
117-clip unanimous test set, and exports the clips where the model disagrees with the consensus
label — the exact set a human needs to listen to for the audit reported in the paper.

**Workflow**
1. Run cells 1–4. Cell 4 prints the confusion matrix; it should reproduce **[[60, 5], [3, 49]]**
   (UAR 0.933) — that confirms you loaded the right model with the right eval setup.
2. Cell 5 writes `audit_disagreements.csv` (the ~8 disagreement clips) to Drive.
3. **Listen** to each clip and fill the `verdict` column with `model_error`, `label_error`, or
   `ambiguous` (and optional `notes`).
4. Run cell 6 to compute the adjudicated accuracy + label-error floor for the paper.

Fully local; no data leaves the machine. Matches `finetune.ipynb` (LABEL2ID {NG:0, PZ:1},
`AutoFeatureExtractor`, librosa 16 kHz, 25 s eval crop).

In [ ]:
# === Cell 1: setup + config ===
%pip -q install "transformers>=4.56,<5" "accelerate>=1.0" "huggingface_hub<1.0" librosa soundfile scikit-learn pandas
from google.colab import drive; drive.mount("/content/drive")

CFG = {
    "BEST_MODEL": "/content/drive/MyDrive/CHANGE_ME/finetune_outputs/best_model_ours-mHuBERT-147",
    "DATASET_CSV":"/content/drive/MyDrive/CHANGE_ME/labeling_outputs/dataset_final.csv",
    "AUDIO_ZIP":  "/content/drive/MyDrive/CHANGE_ME/segments_filtered.zip",
    "OUT_CSV":    "/content/drive/MyDrive/CHANGE_ME/labeling_outputs/audit_disagreements.csv",
    "EVAL_MAX_S": 25.0,     # must match finetune.ipynb
    "BATCH_EVAL": 8,
}
LABEL2ID = {"NG": 0, "PZ": 1}; ID2LABEL = {0: "NG", 1: "PZ"}
import os, glob, zipfile, json, numpy as np, pandas as pd, torch, librosa


In [ ]:
# === Cell 2: extract audio FRESH (matches finetune.ipynb) + robust id -> wav map ===
import unicodedata, shutil
from pathlib import Path
def nfc(s): return unicodedata.normalize("NFC", s)

AUDIO_ROOT = "/content/audit_audio"
shutil.rmtree(AUDIO_ROOT, ignore_errors=True)   # always fresh: avoids a stale/broken prior extraction
os.makedirs(AUDIO_ROOT, exist_ok=True)

def _fix_zip_name(info):                         # same fix finetune.ipynb uses
    if info.flag_bits & 0x800:
        return info.filename
    try:
        return info.filename.encode("cp437").decode("utf-8")
    except (UnicodeDecodeError, UnicodeEncodeError):
        return info.filename

with zipfile.ZipFile(CFG["AUDIO_ZIP"]) as zf:
    for info in zf.infolist():
        if info.is_dir():
            continue
        dest = os.path.join(AUDIO_ROOT, _fix_zip_name(info).lstrip("/"))
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        with zf.open(info) as src_f, open(dest, "wb") as out_f:
            out_f.write(src_f.read())

wavs = glob.glob(os.path.join(AUDIO_ROOT, "**", "*.wav"), recursive=True)
path_by_id = {}
for p in wavs:
    st = Path(p).stem
    path_by_id.setdefault(st, p)
    path_by_id.setdefault(nfc(st), p)
print(len(wavs), "wavs extracted;", len(path_by_id), "lookup keys")
print("sample wav stems:")
for p in wavs[:3]:
    print("   ", repr(Path(p).stem))

def find_wav(seg_id):
    return path_by_id.get(seg_id) or path_by_id.get(nfc(seg_id))


In [ ]:
# === Cell 3: load the saved 0.933 model + feature extractor ===
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
FE = AutoFeatureExtractor.from_pretrained(CFG["BEST_MODEL"])
model = AutoModelForAudioClassification.from_pretrained(CFG["BEST_MODEL"]).eval()
device = "cuda" if torch.cuda.is_available() else "cpu"; model.to(device)
print("loaded:", CFG["BEST_MODEL"], "| device:", device, "| labels:", model.config.id2label)


In [ ]:
# === Cell 4: run inference on the 117 unanimous test clips ===
from sklearn.metrics import confusion_matrix, recall_score
df = pd.read_csv(CFG["DATASET_CSV"], dtype={"id": str})
test = df[df["split"] == "test"].copy()
test["path"] = test["id"].map(find_wav)
missing = int(test["path"].isna().sum())
if missing:
    print(f"[!] {missing}/{len(test)} unmatched. Compare formats below:")
    print("  sample test ids :")
    for i in test["id"].head(3): print("     ", repr(i))
    print("  sample wav keys :")
    for k in list(path_by_id)[:3]: print("     ", repr(k))
assert missing == 0, f"{missing}/{len(test)} test clips have no wav (see printed samples above)"
print(f"{len(test)} test clips matched")

@torch.no_grad()
def predict(paths):
    preds, probs = [], []
    for i in range(0, len(paths), CFG["BATCH_EVAL"]):
        batch = paths[i:i+CFG["BATCH_EVAL"]]
        wavs = [librosa.load(p, sr=16000, mono=True)[0][: int(CFG["EVAL_MAX_S"]*16000)] for p in batch]
        inp = FE(wavs, sampling_rate=16000, return_tensors="pt", padding=True).to(device)
        pr = torch.softmax(model(**inp).logits, -1).cpu().numpy()
        probs.extend(pr); preds.extend(pr.argmax(-1))
    return np.array(preds), np.array(probs)

y_true = test["final_label"].map(LABEL2ID).to_numpy()
y_pred, y_prob = predict(test["path"].tolist())
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
uar = recall_score(y_true, y_pred, average="macro")
print("confusion [[NG->NG, NG->PZ], [PZ->NG, PZ->PZ]] =", cm.tolist())
print(f"UAR = {uar:.4f}   (expect [[60, 5], [3, 49]] , UAR 0.933)")


In [ ]:
# === Cell 4b: full primary-run metrics + 95% bootstrap CI + per-speaker (backs Table 6) ===
from sklearn.metrics import f1_score
def uar_from(idx):
    yt, yh = y_true[idx], y_pred[idx]
    return recall_score(yt, yh, average="macro")

rng = np.random.default_rng(42)          # seed 42, matches finetune.ipynb
n = len(y_true)
boot = np.array([uar_from(rng.integers(0, n, n)) for _ in range(2000)])
ci = (float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5)))

per_role = {}
for role in ["child", "therapist"]:
    m = (test["speaker_role"].to_numpy() == role)
    per_role[role] = {"n": int(m.sum()), "uar": round(float(uar_from(m)), 4)}

primary = {
    "run": "ours-mHuBERT-147 (primary / cross-corpus matrix)",
    "model": CFG["BEST_MODEL"],
    "n": n,
    "uar": round(float(uar), 4),
    "f1_macro": round(float(f1_score(y_true, y_pred, average="macro")), 4),
    "accuracy": round(float((y_true == y_pred).mean()), 4),
    "confusion": cm.tolist(),
    "uar_ci95_bootstrap_seed42": [round(ci[0], 4), round(ci[1], 4)],
    "per_speaker_uar": per_role,
}
out = "/content/drive/MyDrive/CHANGE_ME/finetune_outputs/primary_metrics.json"
json.dump(primary, open(out, "w"), indent=2)
print(json.dumps(primary, indent=2))
print("\nsaved ->", out, "  (add this file to the repo's results/metrics/)")


In [ ]:
# === Cell 5: export disagreement clips for human listening ===
test = test.reset_index(drop=True)
test["model_pred"] = [ID2LABEL[p] for p in y_pred]
test["p_ng"] = y_prob[:, 0].round(3); test["p_pz"] = y_prob[:, 1].round(3)
dis = test[test["final_label"] != test["model_pred"]].copy()
dis = dis.rename(columns={"final_label": "consensus_label"})
dis["verdict"] = ""   # <- FILL: model_error | label_error | ambiguous
dis["notes"]   = ""
cols = ["id", "speaker_role", "duration_s", "consensus_label", "model_pred", "p_ng", "p_pz", "verdict", "notes"]
dis[cols].to_csv(CFG["OUT_CSV"], index=False)
print(f"{len(dis)} disagreements -> {CFG['OUT_CSV']}")
dis[cols]


### Now listen and adjudicate
Open `audit_disagreements.csv`, play each clip (`id` maps to a `.wav` under the extracted audio),
and set `verdict` to `model_error`, `label_error`, or `ambiguous`. Save, then run the cell below.

In [ ]:
# === Cell 6: compute audit results from the filled CSV ===
a = pd.read_csv(CFG["OUT_CSV"])
n_test = 117
vc = a["verdict"].fillna("").str.strip().value_counts()
model_err = int(vc.get("model_error", 0)); label_err = int(vc.get("label_error", 0)); amb = int(vc.get("ambiguous", 0))
assert model_err + label_err + amb == len(a), "some rows still have no verdict"
adjudicated_acc = (n_test - model_err) / n_test          # correcting label errors
label_floor = label_err / n_test
print(f"disagreements audited : {len(a)}")
print(f"  model errors        : {model_err}")
print(f"  label errors        : {label_err}")
print(f"  ambiguous           : {amb}")
print(f"adjudicated accuracy  : {adjudicated_acc:.3f}  ({n_test-model_err}/{n_test})")
print(f"unanimous label-error floor : {label_floor*100:.1f}%")
print("\n--> put these numbers into the paper's Human Audit subsection (Sec. 4.4).")
